# Notebook 2: Cloud-Optimized GeoTIFF (COG)

Open a COG from a URL and read a spatial window without downloading the full file.

**Dependencies:** `rasterio`, `pystac-client`, `planetary-computer`

In [ ]:
import rasterio
from rasterio.windows import from_bounds
import pystac_client
import planetary_computer

## Get a COG URL from STAC

We reuse the STAC search from Notebook 1 to get a single asset href.

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1"
)
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=[-122.4, 37.6, -122.2, 37.8],
    datetime="2023-06-15",
    max_items=1,
)
items = list(search.items())
item = planetary_computer.sign(items[0])
url = item.assets["B04"].href
print("COG URL (B04):", url[:90], "...")

## Open the COG (no full download)

`rasterio.open(url)` uses HTTP range requests; metadata is read first, then only requested windows are transferred.

In [ ]:
with rasterio.open(url) as src:
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Shape (height, width):", src.shape)
    print("Block size:", src.block_shapes)

## Read a window by geographic bounds

Request only a small subregion; only the tiles covering that window are streamed.

In [ ]:
# Small box inside the scene (lon, lat)
left, bottom, right, top = -122.35, 37.65, -122.30, 37.70

with rasterio.open(url) as src:
    window = from_bounds(left, bottom, right, top, src.transform)
    data = src.read(1, window=window)
    print("Window shape:", data.shape)
    print("Data dtype:", data.dtype)

## Optional: quick plot

Visualize the window we read (small subset of the full scene).

In [ ]:
import matplotlib.pyplot as plt

with rasterio.open(url) as src:
    window = from_bounds(left, bottom, right, top, src.transform)
    data = src.read(1, window=window)

fig, ax = plt.subplots(1, 1, figsize=(6, 5))
ax.imshow(data, cmap="gray")
ax.set_title("COG window (B04) — streamed only")
plt.tight_layout()
plt.show()